In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from huutopussi_rl.analysis import (
    HuutopussiSimulation,
    calculate_count_of_certain_cards_in_hand,
    calculate_count_of_certain_cards_in_deal,
    calculate_trump_combos_in_hand,
    probability_of_total_aces_given_hand,
    probability_of_total_trump_combos_given_hand
)
from pathlib import Path

import pandas as pd
import seaborn as sns


CONSTANTS_DIR = Path("../huutopussi_rl/statistical_constants")
aces_probabilities = pd.read_csv(CONSTANTS_DIR / "probability_of_aces_given_hand.csv")
trump_probabilities = pd.read_csv(CONSTANTS_DIR / "probability_of_trumps_given_hand.csv")

In [ ]:
for i in range(5):
    for j in range(5):
        probability = probability_of_total_aces_given_hand(aces_in_hand=j, total_aces=i)
        print(f"Probability of {i} aces given {j} aces in hand: {probability}")

In [ ]:
simulation = HuutopussiSimulation()

In [ ]:
sim_1 = simulation.simulate_deal()

In [ ]:
sim_1

In [ ]:
aces = calculate_count_of_certain_cards_in_deal(sim_1, 13)

In [ ]:
aces

In [ ]:
calculate_trump_combos_in_hand(sim_1["player_1"])

In [ ]:
df = simulation.simulate_statistics(10)

In [ ]:
display(df)

In [ ]:
df.sum(axis=0)

In [ ]:
SIMULATION_COUNT = 10000
df = simulation.simulate_statistics(SIMULATION_COUNT)/(3 * SIMULATION_COUNT) # Multiplied by 3 since there are 3 players in the game.


In [ ]:
ACES_BEFORE = [f"aces_before_devil_{i}" for i in range(0, 5)]
ACES_AFTER = [f"aces_after_devil_{i}" for i in range(0, 5)]
TRUMP_COMBOS_BEFORE = [f"trumps_before_devil_{i}" for i in range(0, 5)]
TRUMP_COMBOS_AFTER = [f"trumps_after_devil_{i}" for i in range(0, 5)]

In [ ]:
values = df[TRUMP_COMBOS_BEFORE].sum(axis=0)

fig, ax = plt.subplots(figsize=(10, 6))

bars = ax.bar(values.index, values.values, label="Aces Before Devil")

ax.bar_label(bars, padding=3, fmt="%.2f")
ax.set_title("Aces Before Devil by Count")
ax.set_xlabel("Number of Aces")
ax.set_ylabel("Frequency")
ax.tick_params(axis="x", rotation=90)

plt.tight_layout()
plt.show()

In [ ]:
values = df[TRUMP_COMBOS_AFTER].sum(axis=0)

fig, ax = plt.subplots(figsize=(10, 6))

bars = ax.bar(values.index, values.values, label="Aces Before Devil")

ax.bar_label(bars, padding=3, fmt="%.2f")
ax.set_title("Aces Before Devil by Count")
ax.set_xlabel("Number of Aces")
ax.set_ylabel("Frequency")
ax.tick_params(axis="x", rotation=90)

plt.tight_layout()
plt.show()

In [ ]:
values = df[ACES_BEFORE].sum(axis=0)

fig, ax = plt.subplots(figsize=(10, 6))

bars = ax.bar(values.index, values.values, label="Aces Before Devil")

ax.bar_label(bars, padding=3, fmt="%.2f")
ax.set_title("Aces Before Devil by Count")
ax.set_xlabel("Number of Aces")
ax.set_ylabel("Frequency")
ax.tick_params(axis="x", rotation=90)

plt.tight_layout()
plt.show()

In [ ]:
values = df[ACES_AFTER].sum(axis=0)

fig, ax = plt.subplots(figsize=(10, 6))

bars = ax.bar(values.index, values.values, label="Aces After Devil")

ax.bar_label(bars, padding=3, fmt="%.2f")
ax.set_title("Aces After Devil by Count")
ax.set_xlabel("Number of Aces")
ax.set_ylabel("Frequency")
ax.tick_params(axis="x", rotation=90)

plt.tight_layout()
plt.show()

In [ ]:
for i in range(5):
    for j in range(5):
        probability = probability_of_total_trump_combos_given_hand(
            complete_combos=i, singleton_halves=j
        )
        print(
            f"Probability of {i} complete combos and {j} singleton halves: {probability}"
        )

In [ ]:
sns.set_theme(style="whitegrid", context="notebook")

fig, axes = plt.subplots(2, 2, figsize=(16, 12), constrained_layout=True)

ace_heatmap = aces_probabilities.set_index("aces_in_hand_before").drop(columns=[])
ace_heatmap.columns = [column.replace("prob_", "").replace("_after", "") for column in ace_heatmap.columns]
sns.heatmap(
    ace_heatmap,
    annot=True,
    fmt=".3f",
    cmap="Blues",
    vmin=0,
    vmax=1,
    ax=axes[0, 0],
)
axes[0, 0].set_title("Aces after devil deck")
axes[0, 0].set_xlabel("Total aces after devil deck")
axes[0, 0].set_ylabel("Aces already in hand")

trump_zero_halves = trump_probabilities[
    trump_probabilities["singleton_halves_before"] == 0
].set_index("complete_combos_before").drop(columns=["singleton_halves_before"])
trump_zero_halves.columns = [
    column.replace("prob_", "").replace("_after", "") for column in trump_zero_halves.columns
]
sns.heatmap(
    trump_zero_halves,
    annot=True,
    fmt=".3f",
    cmap="Oranges",
    vmin=0,
    vmax=1,
    ax=axes[0, 1],
)
axes[0, 1].set_title("Trump combos after: no singleton halves")
axes[0, 1].set_xlabel("Total trump combos after devil deck")
axes[0, 1].set_ylabel("Complete combos already in hand")

trump_melted = trump_probabilities.melt(
    id_vars=["complete_combos_before", "singleton_halves_before"],
    var_name="total_combos_after",
    value_name="probability",
)
trump_melted["total_combos_after"] = (
    trump_melted["total_combos_after"].str.extract(r"(\d+)")[0].astype(int)
)
trump_melted["conditioning_state"] = (
    "complete="
    + trump_melted["complete_combos_before"].astype(str)
    + ", halves="
    + trump_melted["singleton_halves_before"].astype(str)
)

selected_states = [(0, 0), (0, 2), (1, 1), (2, 0), (3, 1), (4, 0)]
selected = trump_melted[
    trump_melted[["complete_combos_before", "singleton_halves_before"]]
    .apply(tuple, axis=1)
    .isin(selected_states)
]
sns.barplot(
    data=selected,
    x="total_combos_after",
    y="probability",
    hue="conditioning_state",
    ax=axes[1, 0],
)
axes[1, 0].set_title("Selected trump-combo conditional distributions")
axes[1, 0].set_xlabel("Total trump combos after devil deck")
axes[1, 0].set_ylabel("Probability")
axes[1, 0].legend(title="Initial hand", fontsize="small")

trump_heatmap = trump_melted.pivot_table(
    index=["complete_combos_before", "singleton_halves_before"],
    columns="total_combos_after",
    values="probability",
)
sns.heatmap(
    trump_heatmap,
    annot=True,
    fmt=".3f",
    cmap="viridis",
    vmin=0,
    vmax=1,
    ax=axes[1, 1],
)
axes[1, 1].set_title("All trump-combo conditional distributions")
axes[1, 1].set_xlabel("Total trump combos after devil deck")
axes[1, 1].set_ylabel("Initial hand: complete combos, singleton halves")

plt.show()